In [3]:
import os
import sys

project_root = os.path.abspath("..")
sys.path.append(project_root)

In [4]:
import os
import sys
import json
import itertools
from copy import deepcopy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# make sure notebook can find src/
sys.path.append(".")

from src.utils.seed import set_seed
from src.data.preprocess import load_data, prepare_lstm_data
from src.data.dataset import LSTMDataset, MovieReviewDataset
from src.data.tokenizer_utils import get_tokenizer
from src.models.lstm_model import LSTMSentimentClassifier  
from src.models.bert_classifier import BertSentimentClassifier

In [5]:
set_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [6]:
DATA_PATH = "../data/IMDB Dataset.csv"
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(40000, 2)


,review,sentiment
0,I really liked this Summerslam due to the look...,positive
1,Not many television shows appeal to quite as m...,positive
2,The film quickly gets to a major chase scene w...,negative
3,Jane Austen would definitely approve of this o...,positive
4,Expectations were somewhat high for me when I ...,negative


In [16]:
def train_one_epoch_lstm(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for features, labels in tqdm(loader, leave=False):
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) >= 0.5).long().cpu().numpy()
        true = labels.long().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(true)

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return avg_loss, acc, f1


@torch.no_grad()
def evaluate_lstm(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    for features, labels in loader:
        features = features.to(device)
        labels = labels.to(device)

        outputs = model(features)
        loss = criterion(outputs, labels)

        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) >= 0.5).long().cpu().numpy()
        true = labels.long().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(true)

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return avg_loss, acc, f1

In [17]:
def run_lstm_experiment(config, df, device, num_epochs=4):
    set_seed(42)

    data_dict, stoi = prepare_lstm_data(
        df=df,
        vocab_size=config["vocab_size"],
        seq_length=config["seq_length"],
        random_state=42,
        remove_stopwords=config["remove_stopwords"]
    )

    train_dataset = LSTMDataset(data_dict["X_train"], data_dict["y_train"])
    val_dataset = LSTMDataset(data_dict["X_val"], data_dict["y_val"])

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False
    )

    model = LSTMSentimentClassifier(
        vocab_size=len(stoi),
        embedding_dim=config["embedding_dim"],
        hidden_dim=config["hidden_dim"],
        dropout_rate=config["dropout_rate"]
    ).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])

    history = {
        "train_loss": [],
        "train_acc": [],
        "train_f1": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": []
    }

    best_val_f1 = -1
    best_state = None

    for epoch in range(num_epochs):
        train_loss, train_acc, train_f1 = train_one_epoch_lstm(
            model, train_loader, optimizer, criterion, device
        )

        val_loss, val_acc, val_f1 = evaluate_lstm(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = deepcopy(model.state_dict())

        print(
            f"Epoch {epoch+1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}"
        )

    result = {
        "config": config,
        "best_val_f1": best_val_f1,
        "final_val_acc": history["val_acc"][-1],
        "final_val_f1": history["val_f1"][-1],
        "history": history,
        "best_state_dict": best_state,
        "stoi_size": len(stoi)
    }

    return result

In [18]:
baseline_config = {
    "vocab_size": 5000,
    "seq_length": 500,
    "remove_stopwords": False,
    "batch_size": 32,
    "embedding_dim": 128,
    "hidden_dim": 128,
    "dropout_rate": 0.3,
    "learning_rate": 1e-3
}

baseline_result = run_lstm_experiment(
    config=baseline_config,
    df=df,
    device=DEVICE,
    num_epochs=4
)

Epoch 1/4 | Train Loss: 0.4779 | Train Acc: 0.7486 | Train F1: 0.7587 | Val Loss: 0.3144 | Val Acc: 0.8645 | Val F1: 0.8629


Epoch 2/4 | Train Loss: 0.2847 | Train Acc: 0.8789 | Train F1: 0.8791 | Val Loss: 0.2759 | Val Acc: 0.8815 | Val F1: 0.8844


Epoch 3/4 | Train Loss: 0.2225 | Train Acc: 0.9099 | Train F1: 0.9100 | Val Loss: 0.2694 | Val Acc: 0.8874 | Val F1: 0.8899


Epoch 4/4 | Train Loss: 0.1679 | Train Acc: 0.9351 | Train F1: 0.9352 | Val Loss: 0.2916 | Val Acc: 0.8866 | Val F1: 0.8905


In [19]:
lstm_param_grid = {
    "vocab_size": [1000, 5000],
    "seq_length": [300, 500],
    "remove_stopwords": [False, True],
    "batch_size": [32],
    "embedding_dim": [128],
    "hidden_dim": [128, 256],
    "dropout_rate": [0.3, 0.5],
    "learning_rate": [1e-3, 5e-4]
}

In [20]:
all_combinations = list(itertools.product(
    lstm_param_grid["vocab_size"],
    lstm_param_grid["seq_length"],
    lstm_param_grid["remove_stopwords"],
    lstm_param_grid["batch_size"],
    lstm_param_grid["embedding_dim"],
    lstm_param_grid["hidden_dim"],
    lstm_param_grid["dropout_rate"],
    lstm_param_grid["learning_rate"]
))

experiment_configs = []
for combo in all_combinations:
    config = {
        "vocab_size": combo[0],
        "seq_length": combo[1],
        "remove_stopwords": combo[2],
        "batch_size": combo[3],
        "embedding_dim": combo[4],
        "hidden_dim": combo[5],
        "dropout_rate": combo[6],
        "learning_rate": combo[7]
    }
    experiment_configs.append(config)

print("Total combinations:", len(experiment_configs))

Total combinations: 64


In [21]:
selected_experiments = [
    {
        "vocab_size": 1000,
        "seq_length": 500,
        "remove_stopwords": True,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 128,
        "dropout_rate": 0.3,
        "learning_rate": 1e-3
    },
    {
        "vocab_size": 5000,
        "seq_length": 500,
        "remove_stopwords": False,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 128,
        "dropout_rate": 0.3,
        "learning_rate": 1e-3
    },
    {
        "vocab_size": 5000,
        "seq_length": 500,
        "remove_stopwords": False,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 256,
        "dropout_rate": 0.3,
        "learning_rate": 1e-3
    },
    {
        "vocab_size": 5000,
        "seq_length": 300,
        "remove_stopwords": False,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 128,
        "dropout_rate": 0.3,
        "learning_rate": 1e-3
    },
    {
        "vocab_size": 5000,
        "seq_length": 500,
        "remove_stopwords": False,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 128,
        "dropout_rate": 0.5,
        "learning_rate": 1e-3
    },
    {
        "vocab_size": 5000,
        "seq_length": 500,
        "remove_stopwords": False,
        "batch_size": 32,
        "embedding_dim": 128,
        "hidden_dim": 128,
        "dropout_rate": 0.3,
        "learning_rate": 5e-4
    }
]

In [22]:
lstm_results = []

for i, config in enumerate(selected_experiments, start=1):
    print("=" * 80)
    print(f"Running LSTM experiment {i}/{len(selected_experiments)}")
    print(config)

    result = run_lstm_experiment(
        config=config,
        df=df,
        device=DEVICE,
        num_epochs=4
    )

    lstm_results.append(result)

Running LSTM experiment 1/6
{'vocab_size': 1000, 'seq_length': 500, 'remove_stopwords': True, 'batch_size': 32, 'embedding_dim': 128, 'hidden_dim': 128, 'dropout_rate': 0.3, 'learning_rate': 0.001}


Epoch 1/4 | Train Loss: 0.4607 | Train Acc: 0.7719 | Train F1: 0.7762 | Val Loss: 0.3554 | Val Acc: 0.8419 | Val F1: 0.8439


Epoch 2/4 | Train Loss: 0.3357 | Train Acc: 0.8527 | Train F1: 0.8541 | Val Loss: 0.3572 | Val Acc: 0.8396 | Val F1: 0.8517


Epoch 3/4 | Train Loss: 0.3056 | Train Acc: 0.8671 | Train F1: 0.8682 | Val Loss: 0.3542 | Val Acc: 0.8414 | Val F1: 0.8303


Epoch 4/4 | Train Loss: 0.2706 | Train Acc: 0.8868 | Train F1: 0.8872 | Val Loss: 0.3346 | Val Acc: 0.8541 | Val F1: 0.8559
Running LSTM experiment 2/6
{'vocab_size': 5000, 'seq_length': 500, 'remove_stopwords': False, 'batch_size': 32, 'embedding_dim': 128, 'hidden_dim': 128, 'dropout_rate': 0.3, 'learning_rate': 0.001}


...

Epoch 4/4 | Train Loss: 0.1900 | Train Acc: 0.9261 | Train F1: 0.9261 | Val Loss: 0.2894 | Val Acc: 0.8825 | Val F1: 0.8855
Running LSTM experiment 6/6
{'vocab_size': 5000, 'seq_length': 500, 'remove_stopwords': False, 'batch_size': 32, 'embedding_dim': 128, 'hidden_dim': 128, 'dropout_rate': 0.3, 'learning_rate': 0.0005}


Epoch 1/4 | Train Loss: 0.5299 | Train Acc: 0.7211 | Train F1: 0.7314 | Val Loss: 0.3705 | Val Acc: 0.8326 | Val F1: 0.8282


Epoch 2/4 | Train Loss: 0.3338 | Train Acc: 0.8556 | Train F1: 0.8566 | Val Loss: 0.3114 | Val Acc: 0.8649 | Val F1: 0.8705


Epoch 3/4 | Train Loss: 0.2714 | Train Acc: 0.8861 | Train F1: 0.8868 | Val Loss: 0.2808 | Val Acc: 0.8789 | Val F1: 0.8790


Epoch 4/4 | Train Loss: 0.2220 | Train Acc: 0.9093 | Train F1: 0.9098 | Val Loss: 0.2805 | Val Acc: 0.8809 | Val F1: 0.8836


In [23]:
lstm_results_df = pd.DataFrame([
    {
        **res["config"],
        "best_val_f1": res["best_val_f1"],
        "final_val_acc": res["final_val_acc"],
        "final_val_f1": res["final_val_f1"],
        "stoi_size": res["stoi_size"]
    }
    for res in lstm_results
])

lstm_results_df = lstm_results_df.sort_values(by="best_val_f1", ascending=False).reset_index(drop=True)
lstm_results_df

,vocab_size,seq_length,remove_stopwords,batch_size,embedding_dim,hidden_dim,dropout_rate,learning_rate,best_val_f1,final_val_acc,final_val_f1,stoi_size
0,5000,500,False,32,128,256,0.3,0.0010,0.895647,0.893625,0.895647,5002
1,5000,500,False,32,128,128,0.3,0.0010,0.890525,0.886625,0.890525,5002
2,5000,500,False,32,128,128,0.5,0.0010,0.885478,0.882500,0.885478,5002
3,5000,500,False,32,128,128,0.3,0.0005,0.883568,0.880875,0.883568,5002
4,5000,300,False,32,128,128,0.3,0.0010,0.882638,0.881625,0.881521,5002
5,1000,500,True,32,128,128,0.3,0.0010,0.855908,0.854125,0.855908,1002


In [24]:
os.makedirs("outputs/experiments", exist_ok=True)

lstm_results_df.to_csv("outputs/experiments/lstm_tuning_results.csv", index=False)
print("Saved LSTM tuning results.")

Saved LSTM tuning results.


In [25]:
best_lstm_row = lstm_results_df.iloc[0].to_dict()
best_lstm_row

{'vocab_size': 5000,
 'seq_length': 500,
 'remove_stopwords': False,
 'batch_size': 32,
 'embedding_dim': 128,
 'hidden_dim': 256,
 'dropout_rate': 0.3,
 'learning_rate': 0.001,
 'best_val_f1': 0.8956468424279583,
 'final_val_acc': 0.893625,
 'final_val_f1': 0.8956468424279583,
 'stoi_size': 5002}

BERT

In [7]:
def encode_labels(labels):
    label_map = {"negative": 0, "positive": 1}
    return [label_map[label] for label in labels]


def encode_reviews(tokenizer, reviews, max_len):
    encodings = tokenizer(
        list(reviews),
        add_special_tokens=True,
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=False,
        return_tensors="pt"
    )
    return encodings

In [8]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
from torch.amp import autocast, GradScaler


def train_one_epoch_bert(model, loader, optimizer, criterion, device, scaler, scheduler=None):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        targets = batch["targets"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
        true = targets.detach().cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(true)

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return avg_loss, acc, f1


@torch.no_grad()
def evaluate_bert(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        targets = batch["targets"].to(device, non_blocking=True)

        with autocast(device_type="cuda", enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, targets)

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        true = targets.cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(true)

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return avg_loss, acc, f1

In [9]:
from sklearn.model_selection import train_test_split


def prepare_bert_data(df):
    X = df["review"].astype(str).to_numpy()
    y = df["sentiment"].to_numpy()

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.4, random_state=42, stratify=y
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    y_train = encode_labels(y_train)
    y_val = encode_labels(y_val)
    y_test = encode_labels(y_test)

    return X_train, X_val, X_test, y_train, y_val, y_test


def build_encoding_cache(tokenizer, X_train, X_val, max_lens):
    cache = {}

    for max_len in max_lens:
        cache[max_len] = {
            "train": encode_reviews(tokenizer, X_train, max_len),
            "val": encode_reviews(tokenizer, X_val, max_len)
        }

    return cache

In [10]:
from transformers import get_linear_schedule_with_warmup
from copy import deepcopy


def run_bert_experiment(config, encoding_cache, y_train, y_val, device, num_epochs=3):
    set_seed(42)

    train_dataset = MovieReviewDataset(
        encoding_cache[config["max_len"]]["train"],
        y_train
    )
    val_dataset = MovieReviewDataset(
        encoding_cache[config["max_len"]]["val"],
        y_val
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=config["batch_size"],
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True
    )

    model = BertSentimentClassifier(
        n_classes=2,
        dropout_rate=config["dropout_rate"]
    ).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config.get("weight_decay", 0.01)
    )

    total_training_steps = len(train_loader) * num_epochs
    warmup_steps = int(0.1 * total_training_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_training_steps
    )

    scaler = GradScaler("cuda", enabled=(device.type == "cuda"))

    history = {
        "train_loss": [],
        "train_acc": [],
        "train_f1": [],
        "val_loss": [],
        "val_acc": [],
        "val_f1": []
    }

    best_val_f1 = -1
    best_state = None

    for epoch in range(num_epochs):
        train_loss, train_acc, train_f1 = train_one_epoch_bert(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scaler=scaler,
            scheduler=scheduler
        )

        val_loss, val_acc, val_f1 = evaluate_bert(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["train_f1"].append(train_f1)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_f1"].append(val_f1)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = deepcopy(model.state_dict())

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Train F1: {train_f1:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}"
        )

    result = {
        "config": config,
        "best_val_f1": best_val_f1,
        "final_val_acc": history["val_acc"][-1],
        "final_val_f1": history["val_f1"][-1],
        "history": history,
        "best_state_dict": best_state
    }

    return result

In [11]:
bert_experiments = [
    {
        "max_len": 128,
        "batch_size": 16,
        "dropout_rate": 0.3,
        "learning_rate": 2e-5,
        "weight_decay": 0.01
    },
    {
        "max_len": 128,
        "batch_size": 16,
        "dropout_rate": 0.3,
        "learning_rate": 3e-5,
        "weight_decay": 0.01
    },
    {
        "max_len": 256,
        "batch_size": 8,
        "dropout_rate": 0.3,
        "learning_rate": 2e-5,
        "weight_decay": 0.01
    }
]

In [12]:
print("Using device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

X_train, X_val, X_test, y_train, y_val, y_test = prepare_bert_data(df)

tokenizer = get_tokenizer()
max_lens = sorted(set(config["max_len"] for config in bert_experiments))
encoding_cache = build_encoding_cache(tokenizer, X_train, X_val, max_lens)

bert_results = []

for i, config in enumerate(bert_experiments, start=1):
    print("=" * 80)
    print(f"Running BERT experiment {i}/{len(bert_experiments)}")
    print(config)

    result = run_bert_experiment(
        config=config,
        encoding_cache=encoding_cache,
        y_train=y_train,
        y_val=y_val,
        device=DEVICE,
        num_epochs=2
    )

    bert_results.append(result)

Using device: cuda
CUDA available: True
Running BERT experiment 1/3
{'max_len': 128, 'batch_size': 16, 'dropout_rate': 0.3, 'learning_rate': 2e-05, 'weight_decay': 0.01}


Epoch 1/2 | Train Loss: 0.3770 | Train Acc: 0.8188 | Train F1: 0.8204 | Val Loss: 0.2857 | Val Acc: 0.8791 | Val F1: 0.8791


Epoch 2/2 | Train Loss: 0.1951 | Train Acc: 0.9236 | Train F1: 0.9241 | Val Loss: 0.2996 | Val Acc: 0.8840 | Val F1: 0.8855
Running BERT experiment 2/3
{'max_len': 128, 'batch_size': 16, 'dropout_rate': 0.3, 'learning_rate': 3e-05, 'weight_decay': 0.01}


Epoch 1/2 | Train Loss: 0.3781 | Train Acc: 0.8202 | Train F1: 0.8209 | Val Loss: 0.2835 | Val Acc: 0.8819 | Val F1: 0.8822


Epoch 2/2 | Train Loss: 0.1807 | Train Acc: 0.9309 | Train F1: 0.9312 | Val Loss: 0.2989 | Val Acc: 0.8890 | Val F1: 0.8899
Running BERT experiment 3/3
{'max_len': 256, 'batch_size': 8, 'dropout_rate': 0.3, 'learning_rate': 2e-05, 'weight_decay': 0.01}


Epoch 1/2 | Train Loss: 0.3051 | Train Acc: 0.8615 | Train F1: 0.8623 | Val Loss: 0.2183 | Val Acc: 0.9129 | Val F1: 0.9135


Epoch 2/2 | Train Loss: 0.1299 | Train Acc: 0.9535 | Train F1: 0.9535 | Val Loss: 0.2316 | Val Acc: 0.9186 | Val F1: 0.9189


In [13]:
import pandas as pd

bert_results_df = pd.DataFrame([
    {
        "max_len": result["config"]["max_len"],
        "batch_size": result["config"]["batch_size"],
        "dropout_rate": result["config"]["dropout_rate"],
        "learning_rate": result["config"]["learning_rate"],
        "best_val_f1": result["best_val_f1"],
        "final_val_acc": result["final_val_acc"],
        "final_val_f1": result["final_val_f1"]
    }
    for result in bert_results
])

bert_summary = bert_results_df.sort_values(by="best_val_f1", ascending=False).reset_index(drop=True)
bert_summary

,max_len,batch_size,dropout_rate,learning_rate,best_val_f1,final_val_acc,final_val_f1
0,256,8,0.3,0.00002,0.918899,0.918625,0.918899
1,128,16,0.3,0.00003,0.889936,0.889000,0.889936
2,128,16,0.3,0.00002,0.885517,0.884000,0.885517


In [14]:
bert_results_df.to_csv("outputs/experiments/bert_tuning_results.csv", index=False)
print("Saved BERT tuning results.")

Saved BERT tuning results.


In [26]:
best_lstm_config = lstm_results_df.iloc[0].to_dict()
best_bert_config = bert_results_df.iloc[0].to_dict()

print("Best LSTM config:")
print(best_lstm_config)

print("\nBest BERT config:")
print(best_bert_config)

Best LSTM config:
{'vocab_size': 5000, 'seq_length': 500, 'remove_stopwords': False, 'batch_size': 32, 'embedding_dim': 128, 'hidden_dim': 256, 'dropout_rate': 0.3, 'learning_rate': 0.001, 'best_val_f1': 0.8956468424279583, 'final_val_acc': 0.893625, 'final_val_f1': 0.8956468424279583, 'stoi_size': 5002}

Best BERT config:
{'max_len': 128.0, 'batch_size': 16.0, 'dropout_rate': 0.3, 'learning_rate': 2e-05, 'best_val_f1': 0.8855169010609425, 'final_val_acc': 0.884, 'final_val_f1': 0.8855169010609425}
